In [ ]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score

In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)

# Data preprocessing

In [3]:
data_path = "../data/Customer-Churn-Records.csv"


def clean_data(data_path):
    df = pd.read_csv(data_path)
    df.drop(columns=["RowNumber", "CustomerId", "Surname", "Complain"], inplace=True)
    df.head()

    return df


def split_data(df):
    # Train/ test stratified split of ratio 0.85/0.15
    labels = df.Exited.values
    del df["Exited"]

    X_train, X_test, y_train, y_test = train_test_split(
        df, labels, test_size=0.15, random_state=seed, stratify=labels
    )

    return X_train, y_train, X_test, y_test


In [4]:
records = clean_data(data_path)
X_train, y_train, X_test, y_test = split_data(records)

# Pipeline

In [5]:
cat = [
    "Geography",
    "Gender",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "Satisfaction Score",
    "Card Type",
]

num = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary", "Point Earned"]

In [6]:
scorer = {"Accuracy": "accuracy", "AUC": "roc_auc", "F1": "f1", "Recall": "recall"}

rfParam = {
    "rf__n_estimators": [100, 150, 200, 400, 600],
    "rf__max_features": ["auto", "log2", None, 4, 6, 8, 10],
    "rf__criterion": ["entropy", "gini"],
    "rf__max_depth": [None, 5, 7, 9, 11],
}

lgbParam = {
    "lgb__boosting_type": ["gbdt", "dart"],
    "lgb__n_estimators": [100, 400, 700, 1000],
    "lgb__learning_rate": [0.01, 0.05, 0.12],
    "lgb__num_leaves": [10, 20, 30, 50, 100],
}

xgbParam = {
    "xgb__eta": [0.01, 0.05, 0.1, 0.2, 0.5],
    "xgb__max_depth": [3, 6, 9, 12],
    "xgb__reg_alpha": [0.001, 0.005, 0.01, 0.05, 0.1, 1],
    "xgb__subsample": [0.5, 0.7, 0.9, 1],
}

In [ ]:
cols_to_drop = ["RowNumber", "CustomerId", "Surname", "Complain"]

preprocessor = ColumnTransformer(
    [
        ("column_dropper", "drop", cols_to_drop),
        ("oh", OneHotEncoder(handle_unknown="ignore"), cat),
        ("scaler", StandardScaler(), num),
    ],
    remainder="passthrough",
)

In [8]:
rfPipe = Pipeline(
    [("pre", preprocessor), ("rf", RandomForestClassifier(random_state=seed))]
)

lgbPipe = Pipeline(
    [("dp", preprocessor), ("lgb", lgb.LGBMClassifier(random_state=seed))]
)

xgbPipe = Pipeline(
    [
        ("dp", preprocessor),
        (
            "xgb",
            xgb.XGBClassifier(random_state=seed, use_label_encoder=False, verbosity=0),
        ),
    ]
)

In [9]:
# A dictionary to loop through pipelines and parameters
pipeParam = {
    "rf": (rfPipe, rfParam),
    "lgb": (lgbPipe, lgbParam),
    "xgb": (xgbPipe, xgbParam),
}

# Model Evaluation and Hyperparameters Tuning

In [10]:
def get_test_score(
    pipe: Pipeline, param: dict, score_function, X_train, X_test, Y_train, Y_test
):
    pipe.set_params(**param)
    pipe.fit(X_train, Y_train)
    y_pred_test = pipe.predict(X_test)
    trialTest = score_function(Y_test, y_pred_test)

    return trialTest

In [11]:
gsBestParams = {}
gsTestResults = {}

# For each model, we want to find the best parameter
# setting each for every metrics.
for key in pipeParam:
    (pipe, param) = pipeParam.get(key)

    gs = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param,
        scoring=scorer,
        cv=StratifiedKFold(5),
        refit=False,
        random_state=seed,
    )

    gs.fit(X_train, y_train)
    results = gs.cv_results_

    # Get best parameters for each metrics
    accuracy_best_index = np.argmin(results["rank_test_Accuracy"])
    accuracy_best_param = results["params"][accuracy_best_index]

    auc_best_index = np.argmin(results["rank_test_AUC"])
    auc_best_param = results["params"][auc_best_index]

    f1_best_index = np.argmin(results["rank_test_F1"])
    f1_best_param = results["params"][f1_best_index]

    recall_best_index = np.argmin(results["rank_test_Recall"])
    recall_best_param = results["params"][recall_best_index]

    aucTest = get_test_score(
        pipe, auc_best_param, roc_auc_score, X_train, X_test, y_train, y_test
    )

    accTest = get_test_score(
        pipe, accuracy_best_param, accuracy_score, X_train, X_test, y_train, y_test
    )

    f1Test = get_test_score(
        pipe, f1_best_param, f1_score, X_train, X_test, y_train, y_test
    )

    recallTest = get_test_score(
        pipe, recall_best_param, recall_score, X_train, X_test, y_train, y_test
    )

    gsTestResults[key] = {
        "auc": aucTest,
        "acc": accTest,
        "f1": f1Test,
        "recall": recallTest,
    }

    gsBestParams[key] = {
        "auc": auc_best_param,
        "acc": accuracy_best_param,
        "f1": f1_best_param,
        "recall": recall_best_param,
    }

/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
5 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/sklearn/base.py

[LightGBM] [Info] Number of positive: 1385, number of negative: 5415
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000620 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1142
[LightGBM] [Info] Number of data points in the train set: 6800, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203676 -> initscore=-1.363473
[LightGBM] [Info] Start training from score -1.363473


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1385, number of negative: 5415
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000487 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1140
[LightGBM] [Info] Number of data points in the train set: 6800, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203676 -> initscore=-1.363473
[LightGBM] [Info] Start training from score -1.363473


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1732, number of negative: 6768
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000486 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1142
[LightGBM] [Info] Number of data points in the train set: 8500, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203765 -> initscore=-1.362929
[LightGBM] [Info] Start training from score -1.362929


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1732, number of negative: 6768
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000475 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1142
[LightGBM] [Info] Number of data points in the train set: 8500, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203765 -> initscore=-1.362929
[LightGBM] [Info] Start training from score -1.362929


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [12]:
gsTestResults

{'rf': {'auc': 0.7164909514894735,
  'acc': 0.8613333333333333,
  'f1': 0.5988483685220729},
 'lgb': {'auc': 0.7216474529510297,
  'acc': 0.8586666666666667,
  'f1': 0.5927342256214149},
 'xgb': {'auc': 0.7203911715439946,
  'acc': 0.8566666666666667,
  'f1': 0.5768500948766604}}

In [13]:
gsBestParams

{'rf': {'auc': {'rf__n_estimators': 600,
   'rf__max_features': 6,
   'rf__max_depth': 9,
   'rf__criterion': 'gini'},
  'acc': {'rf__n_estimators': 150,
   'rf__max_features': 8,
   'rf__max_depth': 11,
   'rf__criterion': 'gini'},
  'f1': {'rf__n_estimators': 100,
   'rf__max_features': None,
   'rf__max_depth': None,
   'rf__criterion': 'gini'},
  'recall': {'rf__n_estimators': 100,
   'rf__max_features': None,
   'rf__max_depth': None,
   'rf__criterion': 'gini'}},
 'lgb': {'auc': {'lgb__num_leaves': 10,
   'lgb__n_estimators': 700,
   'lgb__learning_rate': 0.01,
   'lgb__boosting_type': 'gbdt'},
  'acc': {'lgb__num_leaves': 10,
   'lgb__n_estimators': 700,
   'lgb__learning_rate': 0.01,
   'lgb__boosting_type': 'gbdt'},
  'f1': {'lgb__num_leaves': 20,
   'lgb__n_estimators': 400,
   'lgb__learning_rate': 0.05,
   'lgb__boosting_type': 'gbdt'},
  'recall': {'lgb__num_leaves': 10,
   'lgb__n_estimators': 1000,
   'lgb__learning_rate': 0.12,
   'lgb__boosting_type': 'gbdt'}},
 'xgb':